# Ordered probit

Generate an ordinal outcome with Gaussian errors and estimate an ordered-probit model.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    CrossNest,
    CrossNestedLogit,
    ErrorComponent,
    ErrorComponentsLogit,
    HybridChoiceModel,
    LatentClassLogit,
    LatentVariable,
    MixedLogit,
    MultinomialLogit,
    Nest,
    NestedLogit,
    OrderedChoiceDataset,
    OrderedLogit,
    OrderedProbit,
    RandomCoefficient,
    ScaledMultinomialLogit,
    UtilitySpec,
    WTPCoefficient,
    WTPMixedLogit,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


def show_result(result):
    print(f"Final log likelihood: {result.loglike:.6f}")
    print(f"AIC: {result.aic:.3f}; BIC: {result.bic:.3f}")
    return pd.DataFrame(
        {
            "estimate": result.values,
            "std. error": result.bse,
            "z": result.zvalues,
            "p-value": result.pvalues,
        },
        index=pd.Index(result.param_names, name="parameter"),
    ).round(6)


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(19)
n_obs = 500
x = rng.normal(size=n_obs)
latent_utility = 0.90 * x + rng.normal(size=n_obs)
y = np.digitize(latent_utility, bins=[-1.0, 0.0, 1.0])

data = OrderedChoiceDataset(
    y=torch.as_tensor(y, dtype=torch.long),
    x={"x": torch.as_tensor(x)},
    weights=torch.ones(n_obs),
    categories=[0, 1, 2, 3],
)
print(pd.Series(y).value_counts().sort_index())


0    123
1    145
2    125
3    107
Name: count, dtype: int64


In [3]:
latent = Beta("B_X", init=0.50) * "x"
thresholds = [
    Beta("TAU_1", init=-0.80),
    Beta("TAU_2", init=0.00),
    Beta("TAU_3", init=0.80),
]
model = OrderedProbit(
    latent,
    thresholds,
    device=device,
    max_iter=60,
)
result = model.fit(data)
show_result(result)


Final log likelihood: -561.267560
AIC: 1130.535; BIC: 1147.394


,estimate,std. error,z,p-value
parameter,,,,
B_X,0.953688,0.063513,15.015652,0.000000
TAU_1,-0.954863,0.072103,-13.243033,0.000000
TAU_2,0.099014,0.063007,1.571474,0.116073
TAU_3,1.069631,0.075425,14.181352,0.000000
